# MarkItDown — PDF → Markdown (local, pure-Python)

Experiment with [**MarkItDown**](https://github.com/microsoft/markitdown) (Microsoft): a
lightweight utility that converts **PDF / Office (DOCX·PPTX·XLSX) / HTML / images / audio / EPub /
CSV / JSON / ZIP …** into Markdown for LLM consumption. Its strength is **breadth of input
formats**, not deep PDF layout analysis.

For **PDFs** (all of this repo's `../../pdf_resources/` are PDFs) MarkItDown extracts the text
layer via **`pdfminer.six`** — fast, offline, no models. Like PyMuPDF4LLM it does **no OCR** by
default and does **not** extract embedded images, so a scanned page yields almost nothing.

Where it sits among the three notebooks in this repo:

| approach | needs | speed | tables | scanned PDFs |
|---|---|---|---|---|
| MinerU (`../mineru`) | layout+OCR+table/formula models | ~15–240 s/doc | ✅ strong | ✅ OCRs |
| PyMuPDF4LLM (`../pymupdf4llm`) | pure Python | ~0.1–2 s/doc | ✅ good + image extraction | ❌ text layer only |
| **MarkItDown (this notebook)** | **pure Python (pdfminer)** | **~0.05–1 s/doc** | ⚠️ weak / linear | ❌ text layer only |

**Optional cloud/LLM upgrades** (not wired here — this run stays offline): pass an `llm_client`
(Azure OpenAI / OpenAI / a Claude client) to caption images, a `docintel_endpoint` for **Azure
Document Intelligence** OCR + table structure, or a `cu_endpoint` for **Azure Content
Understanding**. The manifest columns match the other two notebooks so the three `*_manifest.csv`
files diff 1:1.

## 0. Environment (read first — one-time setup)

MarkItDown needs **Python 3.10+**; we use a dedicated 3.12 venv to match the other notebooks and
launch Jupyter cleanly. From the repo root, in a terminal:

```bash
# 1. a modern Python (Homebrew)
brew install python@3.12

# 2. a venv just for MarkItDown
/opt/homebrew/bin/python3.12 -m venv .venv-markitdown
source .venv-markitdown/bin/activate

# 3. MarkItDown (all format converters) + pypdf (page slicing) — pure Python, no GPU
python -m pip install --upgrade pip
pip install -U "markitdown[all]" pypdf

# 4. make this venv a Jupyter kernel, then launch
pip install jupyter ipykernel pandas markdown
python -m ipykernel install --user --name markitdown --display-name "Python (markitdown)"
jupyter lab   # or: jupyter notebook
```

Then open this notebook and pick the **Python (markitdown)** kernel (top-right kernel picker).


There is **no model download** — extraction reads the PDF text layer, so this notebook runs fully
offline from the first cell.

## 1. Sanity check — right Python, libraries importable

In [1]:
import sys, platform, logging, warnings

print('Python :', sys.version.split()[0], '(need 3.10+)')
assert sys.version_info[:2] >= (3, 10), (
    'Select the "Python (markitdown)" kernel — see section 0.'
)

# pdfminer is chatty about damaged font metadata; quiet it so the log stays readable.
logging.getLogger('pdfminer').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

import markitdown, pypdf
from markitdown import MarkItDown
print('markitdown:', markitdown.__version__)
print('pypdf     :', pypdf.__version__)
print('machine   :', platform.platform())
print('note: pure-Python, CPU only, offline — no OCR unless you wire an LLM / Azure endpoint')

Python : 3.12.13 (need 3.10+)


markitdown: 0.1.6
pypdf     : 6.14.2
machine   : macOS-26.5.2-arm64-arm-64bit
note: pure-Python, CPU only, offline — no OCR unless you wire an LLM / Azure endpoint


## 2. Config

Everything is relative to this notebook's folder (`notebooks/markitdown/`), so the PDFs resolve to
`ocr_exploration/pdf_resources/`. Defaults mirror the other runs so the manifests line up.

In [2]:
from pathlib import Path

HERE      = Path.cwd()
# Robust to being launched from repo root or from notebooks/markitdown/.
REPO_ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'pdf_resources').is_dir()), HERE)
PDF_DIR   = REPO_ROOT / 'pdf_resources'
OUT_DIR   = HERE / 'out'
HTML_DIR  = HERE / 'html'

# MarkItDown has NO page-range option, so to compare fairly with the mineru /
# pymupdf4llm runs we slice each PDF to the first MAX_PAGES pages (via pypdf)
# before converting. None -> convert the whole document.
MAX_PAGES = 15            # matches the other two notebooks; None -> all pages
MAX_FILES = None          # e.g. 2 -> smoke-test the first 2 PDFs; None -> all

OUT_DIR.mkdir(exist_ok=True)
HTML_DIR.mkdir(exist_ok=True)
print('PDF_DIR :', PDF_DIR)
print('OUT_DIR :', OUT_DIR)
print('max_pages/max_files:', MAX_PAGES, MAX_FILES)

PDF_DIR : /Users/asikifthakerhamim/Documents/ocr_exploration/pdf_resources
OUT_DIR : /Users/asikifthakerhamim/Documents/ocr_exploration/notebooks/markitdown/out
max_pages/max_files: 15 None


## 3. Discover the PDFs

In [3]:
pdfs = sorted(PDF_DIR.glob('*.pdf'))
if MAX_FILES:
    pdfs = pdfs[:MAX_FILES]
assert pdfs, f'No PDFs under {PDF_DIR}'

for p in pdfs:
    print(f'{p.stat().st_size/1024:8.0f} KB  {p.name}')
print(f'\n{len(pdfs)} PDF(s)')

     787 KB  AF SE 2.pdf
     206 KB  Admin Eval Framework.docx.pdf
     666 KB  Agenda_ Operational Leadership Institute (SY 26-27).pdf
     274 KB  Athlos 2024-2025 Observation Rubric Teacher.docx.pdf
    2076 KB  Atlanta, GA Code of Ordinances.pdf
     224 KB  Benefits_HR Specialist Evaluation Form Template  (1).pdf
     224 KB  Benefits_HR Specialist Evaluation Form Template .pdf
      81 KB  Bullseye Math Checklist  - Sheet1.pdf
    2431 KB  Classroom Environment Checklist 6-8.docx.pdf
     130 KB  Marzano Rubrics with scales. potential evidence, and elements 2022.pdf

10 PDF(s)


## 4. Convert each PDF with MarkItDown

`MarkItDown().convert(path).text_content` returns the whole Markdown string. Because MarkItDown
can't limit pages, we first slice to the first `MAX_PAGES` pages with `pypdf` into a temp file,
then convert that. We time the conversion and write `out/<safe>/<safe>.md` (folder names are
sanitized so spaces/parens don't bite downstream tools).

No model download, no network — expect **tens to a few hundred ms** per document.

In [4]:
import time, tempfile, os
from markitdown import MarkItDown
from pypdf import PdfReader, PdfWriter

MD = MarkItDown()   # offline: no llm_client / no Azure endpoint

def safe_name(stem: str) -> str:
    for a, b in [('(', '-'), (')', '-'), ('[', '-'), (']', '-'), (' ', '_')]:
        stem = stem.replace(a, b)
    for dash in '\u2010\u2011\u2012\u2013\u2014\u2015\u2212':
        stem = stem.replace(dash, '-')
    return stem

def sliced_pdf(src: Path, n_pages: int, tmpdir: str):
    '''Write the first n_pages of src to a temp PDF; return (path, total_pages, used_pages).'''
    reader = PdfReader(str(src))
    total = len(reader.pages)
    if n_pages is None or total <= n_pages:
        return str(src), total, total
    writer = PdfWriter()
    for pg in reader.pages[:n_pages]:
        writer.add_page(pg)
    dst = os.path.join(tmpdir, 'sliced.pdf')
    with open(dst, 'wb') as f:
        writer.write(f)
    return dst, total, n_pages

def convert(pdf: Path) -> dict:
    stem = pdf.stem
    safe = safe_name(stem)
    out  = OUT_DIR / safe
    out.mkdir(parents=True, exist_ok=True)

    md_path, used_pages, total_pages, err = '', 0, 0, ''
    t0 = time.perf_counter()
    try:
        with tempfile.TemporaryDirectory() as td:
            src, total_pages, used_pages = sliced_pdf(pdf, MAX_PAGES, td)
            text = MD.convert(src).text_content
        dt = time.perf_counter() - t0
        ok = True
        (out / f'{safe}.md').write_text(text, encoding='utf-8')
        md_path = str(out / f'{safe}.md')
    except Exception as e:
        dt = time.perf_counter() - t0
        ok, err = False, repr(e)
        print(f'  !! {stem} failed: {err}')

    return {'stem': stem, 'safe': safe, 'ok': ok, 'latency_s': round(dt, 3),
            'pages': used_pages, 'total_pages': total_pages, 'md_path': md_path, 'err': err}

runs = []
for i, pdf in enumerate(pdfs, 1):
    print(f'[{i}/{len(pdfs)}] {pdf.name} ...', flush=True)
    r = convert(pdf)
    if r['ok']:
        cap = f" (of {r['total_pages']})" if r['pages'] != r['total_pages'] else ''
        print(f'      done in {r["latency_s"]}s — {r["pages"]} page(s){cap}')
    runs.append(r)
print('\nconversion pass complete')

[1/10] AF SE 2.pdf ...


      done in 0.517s — 15 page(s) (of 37)
[2/10] Admin Eval Framework.docx.pdf ...


      done in 0.369s — 7 page(s)
[3/10] Agenda_ Operational Leadership Institute (SY 26-27).pdf ...


      done in 0.149s — 6 page(s)
[4/10] Athlos 2024-2025 Observation Rubric Teacher.docx.pdf ...


      done in 0.583s — 13 page(s)
[5/10] Atlanta, GA Code of Ordinances.pdf ...


...

      done in 0.344s — 10 page(s)
[7/10] Benefits_HR Specialist Evaluation Form Template .pdf ...


      done in 0.326s — 10 page(s)
[8/10] Bullseye Math Checklist  - Sheet1.pdf ...


      done in 0.043s — 15 page(s) (of 33)
[9/10] Classroom Environment Checklist 6-8.docx.pdf ...


      done in 0.194s — 5 page(s)
[10/10] Marzano Rubrics with scales. potential evidence, and elements 2022.pdf ...


      done in 0.525s — 10 page(s)

conversion pass complete


## 5. Collect outputs & compute per-PDF stats

MarkItDown returns one flat Markdown string (no per-block metadata), so we derive the **same
columns the other manifests use** straight from the text: GFM table count from separator rows
(`|---|`), image count from `![...]()` refs, and a paragraph-block count for `n_text`.
`n_equations` is always 0 (MarkItDown has no formula extraction).

In [5]:
import re

TABLE_SEP = re.compile(r'(?m)^\s*\|?[\s:|-]*-{3,}[\s:|-]*\|?\s*$')
IMG_REF   = re.compile(r'!\[[^\]]*\]\([^)]*\)')

def text_stats(text: str):
    n_tables = len(TABLE_SEP.findall(text))
    n_images = len(IMG_REF.findall(text))
    blocks = [b for b in re.split(r'\n\s*\n', text) if b.strip()]
    n_text = sum(1 for b in blocks if '|' not in b.splitlines()[0])
    return n_tables, n_images, n_text

records = []
for r in runs:
    text = Path(r['md_path']).read_text(encoding='utf-8') if r['md_path'] else ''
    n_tables, n_images, n_text = text_stats(text)
    records.append({
        'pdf': r['stem'],
        'ok': r['ok'],
        'backend': 'markitdown',
        'latency_s': r['latency_s'],
        'pages': r['pages'],
        'chars': len(text),
        'n_tables': n_tables,
        'n_images': n_images,
        'n_equations': 0,
        'n_text': n_text,
        'md_path': r['md_path'],
    })

try:
    import pandas as pd
    df = pd.DataFrame(records)
    display(df.drop(columns=['md_path']))
except Exception:
    df = None
    for rec in records:
        print(rec)

,pdf,ok,backend,latency_s,pages,chars,n_tables,n_images,n_equations,n_text
0,AF SE 2,True,markitdown,0.517,15,21837,56,0,0,15
1,Admin Eval Framework.docx,True,markitdown,0.369,7,10487,0,0,0,150
2,Agenda_ Operational Leadership Institute (SY 2...,True,markitdown,0.149,6,7817,32,0,0,18
3,Athlos 2024-2025 Observation Rubric Teacher.docx,True,markitdown,0.583,13,20775,3,0,0,15
4,"Atlanta, GA Code of Ordinances",True,markitdown,0.741,15,20608,0,0,0,405
5,Benefits_HR Specialist Evaluation Form Templat...,True,markitdown,0.344,10,12446,4,0,0,11
6,Benefits_HR Specialist Evaluation Form Template,True,markitdown,0.326,10,12446,4,0,0,11
7,Bullseye Math Checklist - Sheet1,True,markitdown,0.043,15,1302,6,0,0,1
8,Classroom Environment Checklist 6-8.docx,True,markitdown,0.194,5,7476,16,0,0,3
9,Marzano Rubrics with scales. potential evidenc...,True,markitdown,0.525,10,34401,16,0,0,16


## 6. Markdown → standalone HTML

Render each `.md` to a full HTML page using the **same house CSS** as the other notebooks in this
repo. (MathJax is included for parity, though MarkItDown emits no LaTeX; MarkItDown also doesn't
extract images from PDFs, so there are normally no local image refs to rewrite.)

In [6]:
import markdown as md_lib

HOUSE_CSS = (
    'body{font-family:Arial,Helvetica,sans-serif;max-width:900px;margin:24px auto;'
    'padding:0 16px;line-height:1.5}'
    'table{border-collapse:collapse;margin:12px 0}'
    'td,th{border:1px solid #888;padding:4px 8px;vertical-align:top}'
    'h1,h2,h3{margin:.6em 0 .3em}img{max-width:100%}'
)
MATHJAX = (
    "<script>window.MathJax={tex:{inlineMath:[['$','$'],['\\\\(','\\\\)']],"
    "displayMath:[['$$','$$'],['\\\\[','\\\\]']]}};</script>"
    '<script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js"></script>'
)

def to_html(md_path: Path, stem: str) -> Path:
    raw  = md_path.read_text(encoding='utf-8')
    body = md_lib.markdown(raw, extensions=['tables', 'fenced_code'])
    html = (f'<!doctype html><html><head><meta charset="utf-8">'
            f'<title>{stem}</title><style>{HOUSE_CSS}</style>{MATHJAX}'
            f'</head><body>{body}</body></html>')
    out = HTML_DIR / f'{stem}.html'
    out.write_text(html, encoding='utf-8')
    return out

for rec in records:
    if rec['md_path']:
        to_html(Path(rec['md_path']), rec['pdf'])
print('wrote', len(list(HTML_DIR.glob('*.html'))), 'HTML pages to', HTML_DIR)

wrote 10 HTML pages to /Users/asikifthakerhamim/Documents/ocr_exploration/notebooks/markitdown/html


## 7. Preview a result inline

Change `PREVIEW` to any PDF stem from the table above.

In [8]:
from IPython.display import HTML, display

PREVIEW = records[0]['pdf'] if records else None
page = HTML_DIR / f'{PREVIEW}.html'
if PREVIEW and page.exists():
    print(page)
    display(HTML(page.read_text(encoding='utf-8')))
else:
    print('nothing to preview')

/Users/asikifthakerhamim/Documents/ocr_exploration/notebooks/markitdown/html/AF SE 2.html
